In [ ]:
!pip install pandas
!pip install networkx
!pip install kagglehub


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import networkx as nx
# import kagglehub

In [3]:
# path = kagglehub.dataset_download("aramacus/companion-plants")
# print("Path to dataset files:", path)

path = "C:/Users/user/.cache/kagglehub/datasets/aramacus/companion-plants/versions/3"
df = pd.read_csv(f"{path}/companion_plants.csv")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 995 entries, 0 to 994
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Source Node       995 non-null    str  
 1   Link              995 non-null    str  
 2   Destination Node  995 non-null    str  
 3   Source Type       995 non-null    str  
dtypes: str(4)
memory usage: 31.2 KB
None


In [8]:
df.dropna()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 995 entries, 0 to 994
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Source Node       995 non-null    str  
 1   Link              995 non-null    str  
 2   Destination Node  995 non-null    str  
 3   Source Type       995 non-null    str  
dtypes: str(4)
memory usage: 31.2 KB
None


In [12]:
duplicated = df.duplicated()
for index, is_duplicated in enumerate(duplicated):
    if is_duplicated:
        print(f"Row {index} is duplicated: {is_duplicated}")

df.drop_duplicates(inplace = True)
duplicated = df.duplicated()
print("After dropping duplicates:")
for index, is_duplicated in enumerate(duplicated):
    if is_duplicated:
        print(f"Row {index} is duplicated: {is_duplicated}")

Row 326 is duplicated: True
Row 328 is duplicated: True
Row 365 is duplicated: True
Row 454 is duplicated: True
Row 513 is duplicated: True
After dropping duplicates:


In [ ]:
# save the cleaned dataframe
df.to_csv("../data/cleaned_data.csv", index=False)

## Using NetworkX to Model Relationships

In [2]:
df = pd.read_csv("cleaned_data.csv")

In [19]:
# Names of the plants -> nodes
# Edges -> colour-coded, green for help and red for avoid
G = nx.DiGraph()
num_node = 0

for index, row in df.iterrows():
    plant1 = row['Source Node']
    plant2 = row['Destination Node']
    relationship = row['Link']

    if relationship == 'helps' or relationship == 'helped_by':
        color = 'green'
    elif relationship == 'avoid':
        color = 'red'
    else:
        color = 'gray'  # For any other relationships

    name = nx.get_node_attributes(G, "data")
    name_list = list(name.values())

    if plant1 in name_list:
        plant1_node = name_list.index(plant1)
    else:
        G.add_node(num_node, data=plant1)
        plant1_node = num_node
        num_node += 1    
    if plant2 in name_list:
        plant2_node = name_list.index(plant2)
    else:
        G.add_node(num_node, data=plant2)
        plant2_node = num_node
        num_node += 1
        
    G.add_edge(plant1_node, plant2_node, color=color)

In [5]:
name = nx.get_node_attributes(G, "data")
name_list = list(name.values())
print(name_list.index("alliums"))


0


In [14]:
# Recommender function
def recommend_plants(plant):
    # Get the plant node index
    name = nx.get_node_attributes(G, "data")
    index = (list(name.values()).index(plant))

    # Get neighbours of the plant node
    neighbours = list(G.neighbors(index))
    # Filter only the neighbours with green-coloured edges
    recommended_plants = []
    for neighbour in neighbours:
        edge_color = G.get_edge_data(index, neighbour)['color']
        if edge_color == 'green':
            recommended_plants.append(name[neighbour])

    return recommended_plants
    
def avoid_plants(plant):
    # Get the plant node index
    name = nx.get_node_attributes(G, "data")
    index = (list(name.values()).index(plant))

    # Get neighbours of the plant node
    neighbours = list(G.neighbors(index))
    # Filter only the neighbours with red-coloured edges
    avoid_plants = []
    for neighbour in neighbours:
        edge_color = G.get_edge_data(index, neighbour)['color']
        if edge_color == 'red':
            avoid_plants.append(name[neighbour])

    return avoid_plants




In [23]:
print(recommend_plants("beets"))
print(avoid_plants("beets"))

['broccoli', 'beans, bush', 'cabbages', 'lettuce', 'kohlrabi', 'onions', 'brassicas', 'passion fruit', 'catnip', 'garlic', 'mints']
['runner', 'beans, pole']
